# Dual-Track GBDT Training, Calibration & SHAP Explainability (Kaggle)
## Metro Nashville Accounts Payable Late Payment & Cash Outflow Forecasting

This notebook is **self-contained** for Kaggle GPU/CPU execution:
- Automatically discovers `/kaggle/input/*/train.parquet` (`val.parquet`, `test.parquet`), with local fallback to `./data/processed/`.
- **Track 1 (Classification):** Predicts `is_late` (`days_to_pay > 30`) with LightGBM & XGBoost, tunes hyperparameters on `val`, and applies post-hoc **Isotonic Calibration** on `val`.
- **Track 2 (Regression & Cashflow):** Predicts `log1p(days_to_pay)`, inverts via `clip(expm1(y), 0)`, and evaluates pointwise MAE/RMSE/MedianAE plus 7d/14d/30d cash outflow MAPE%.
- **Explainability & Exports:** Computes `shap.TreeExplainer` values on a 1,000-row test sample and exports `gbdt_metrics.json`, `test_predictions.parquet`, `*.joblib`, and PNG diagnostics to `/kaggle/working/artifacts/`.

In [ ]:
import inspect
import json
import logging
from pathlib import Path
from typing import Any

import joblib
import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.preprocessing import OrdinalEncoder

try:
    from sklearn.frozen import FrozenEstimator
except ImportError:
    FrozenEstimator = None

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger("kaggle_gbdt_training")

def locate_data_dir() -> Path:
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        matches = sorted(kaggle_root.rglob("train.parquet"))
        if matches:
            return matches[0].parent
    for local_candidate in (Path("data/processed"), Path("../data/processed")):
        if (local_candidate / "train.parquet").exists():
            return local_candidate
    raise FileNotFoundError("Could not locate train.parquet under /kaggle/input or ./data/processed")

DATA_DIR = locate_data_dir()
if Path("/kaggle/working").exists():
    OUT_DIR = Path("/kaggle/working/artifacts")
elif DATA_DIR == Path("../data/processed"):
    OUT_DIR = Path("../artifacts")
else:
    OUT_DIR = Path("artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_parquet(DATA_DIR / "train.parquet")
val_df = pd.read_parquet(DATA_DIR / "val.parquet")
test_df = pd.read_parquet(DATA_DIR / "test.parquet")

logger.info("Loaded splits from %s -> train=%d, val=%d, test=%d", DATA_DIR, len(train_df), len(val_df), len(test_df))
logger.info(
    "Late rates -> train=%.4f, val=%.4f, test=%.4f",
    train_df["is_late"].mean(),
    val_df["is_late"].mean(),
    test_df["is_late"].mean(),
)

In [ ]:
NUMERIC_FEATURES: tuple[str, ...] = (
    "log_invoice_amount",
    "line_item_count",
    "dept_count",
    "amount_vs_vendor_median_ratio",
    "vendor_prior_paid_count",
    "vendor_prior_late_rate_smoothed",
    "vendor_prior_avg_days_90d",
    "vendor_prior_avg_days_365d",
    "vendor_prior_std_days_365d",
    "vendor_has_prior_credit",
    "dept_inv_count_7d",
    "dept_inv_count_30d",
    "dept_prior_late_rate",
    "same_day_vendor_inv_count",
    "inv_month",
    "inv_quarter",
    "inv_dayofweek",
    "days_to_fiscal_year_end",
)
CATEGORICAL_FEATURES: tuple[str, ...] = ("operating_unit", "primary_department")
FEATURE_COLS: tuple[str, ...] = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET_CLASSIFICATION = "is_late"
TARGET_REGRESSION = "days_to_pay"

FORBIDDEN_COLS = {"payment_date", "payment_number", "days_to_pay", "is_late"}
assert not (set(FEATURE_COLS) & FORBIDDEN_COLS), "Leakage detected in FEATURE_COLS!"

def inverse_transform_days(log_preds: np.ndarray) -> np.ndarray:
    return np.clip(np.expm1(np.asarray(log_preds, dtype=np.float64)), a_min=0.0, a_max=None)

def evaluate_classification(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    invoice_amounts: np.ndarray,
    top_k_fractions: tuple[float, ...] = (0.05, 0.10, 0.20),
    cost_ratios: tuple[float, ...] = (1.0, 5.0, 10.0),
) -> dict[str, Any]:
    y_true_arr = np.asarray(y_true).astype(int)
    y_prob_arr = np.asarray(y_prob, dtype=np.float64)
    amts_arr = np.asarray(invoice_amounts, dtype=np.float64)
    n_samples = len(y_true_arr)
    if len(np.unique(y_true_arr)) < 2:
        pr_auc, roc_auc = float("nan"), float("nan")
    else:
        pr_auc = float(average_precision_score(y_true_arr, y_prob_arr))
        roc_auc = float(roc_auc_score(y_true_arr, y_prob_arr))
    brier_score = float(brier_score_loss(y_true_arr, y_prob_arr))
    n_pos = int(np.sum(y_true_arr))
    base_rate = float(n_pos / n_samples) if n_samples > 0 else 0.0
    total_late_amount = float(np.sum(amts_arr[y_true_arr == 1])) if n_pos > 0 else 0.0
    order = np.argsort(-y_prob_arr, kind="mergesort")
    sorted_y, sorted_amts = y_true_arr[order], amts_arr[order]
    top_k: dict[str, dict[str, Any]] = {}
    sensitivity: list[dict[str, Any]] = []
    for frac in top_k_fractions:
        k = min(n_samples, max(1, int(round(n_samples * frac))))
        top_y, top_amt = sorted_y[:k], sorted_amts[:k]
        tp = int(np.sum(top_y))
        precision = float(tp / k) if k > 0 else 0.0
        recall = float(tp / n_pos) if n_pos > 0 else 0.0
        lift = float(precision / base_rate) if base_rate > 0.0 else 0.0
        captured_amt = float(np.sum(top_amt[top_y == 1]))
        val_captured = float(captured_amt / total_late_amount) if total_late_amount > 0.0 else 0.0
        key = f"{frac:g}"
        top_k[key] = {
            "k": k,
            "review_fraction": float(frac),
            "precision": precision,
            "recall": recall,
            "lift": lift,
            "late_value_captured": val_captured,
            "captured_late_amount": captured_amt,
            "total_late_amount": total_late_amount,
        }
        for ratio in cost_ratios:
            sensitivity.append({
                "review_fraction": float(frac),
                "cost_ratio": float(ratio),
                "k": k,
                "true_positives": tp,
                "false_positives": k - tp,
                "late_value_captured_pct": val_captured * 100.0,
                "net_benefit_score": float((tp * ratio) - k),
                "roi": float((tp * ratio) / k) if k > 0 else 0.0,
            })
    return {
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "brier_score": brier_score,
        "base_rate": base_rate,
        "top_k": top_k,
        "lift_at_k": {f"{f:g}": top_k[f"{f:g}"]["lift"] for f in top_k_fractions},
        "precision_at_k": {f"{f:g}": top_k[f"{f:g}"]["precision"] for f in top_k_fractions},
        "late_value_captured_at_k": {f"{f:g}": top_k[f"{f:g}"]["late_value_captured"] for f in top_k_fractions},
        "business_impact_sensitivity": sensitivity,
    }

def evaluate_regression_and_cashflow(
    df_eval: pd.DataFrame,
    pred_days: np.ndarray,
    horizons: tuple[int, ...] = (7, 14, 30),
) -> dict[str, Any]:
    actual_days = np.asarray(df_eval["days_to_pay"], dtype=np.float64)
    pred_arr = np.asarray(pred_days, dtype=np.float64)
    inv_date = pd.to_datetime(df_eval["invoice_date"])
    pay_date = pd.to_datetime(df_eval["payment_date"])
    actual_diff = (pay_date - inv_date).dt.days
    pred_diff = ((inv_date + pd.to_timedelta(np.round(pred_arr), unit="D")) - inv_date).dt.days
    amts = np.asarray(df_eval["invoice_amount"], dtype=np.float64)
    cashflow_horizons: dict[str, dict[str, Any]] = {}
    for h in horizons:
        actual_out = float(np.sum(amts[actual_diff <= h]))
        pred_out = float(np.sum(amts[pred_diff <= h]))
        abs_err = abs(actual_out - pred_out)
        mape = float((abs_err / actual_out) * 100.0) if actual_out > 0 else (0.0 if pred_out == 0 else 100.0)
        cashflow_horizons[f"{h}d"] = {
            "horizon_days": int(h),
            "actual_outflow": actual_out,
            "predicted_outflow": pred_out,
            "abs_error": abs_err,
            "mape_pct": mape,
        }
    return {
        "mae": float(mean_absolute_error(actual_days, pred_arr)),
        "rmse": float(np.sqrt(mean_squared_error(actual_days, pred_arr))),
        "median_ae": float(median_absolute_error(actual_days, pred_arr)),
        "cashflow_horizons": cashflow_horizons,
    }

In [ ]:
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    encoded_missing_value=-1,
    dtype=np.float64,
)
encoder.fit(train_df[list(CATEGORICAL_FEATURES)].astype(object).fillna("__MISSING__"))

def encode_features(df: pd.DataFrame) -> pd.DataFrame:
    num_part = df[list(NUMERIC_FEATURES)].astype(np.float64).copy()
    cat_raw = df[list(CATEGORICAL_FEATURES)].astype(object).fillna("__MISSING__")
    cat_part = pd.DataFrame(
        encoder.transform(cat_raw),
        columns=list(CATEGORICAL_FEATURES),
        index=df.index,
        dtype=np.float64,
    )
    return pd.concat([num_part, cat_part], axis=1)[list(FEATURE_COLS)]

X_train, X_val, X_test = encode_features(train_df), encode_features(val_df), encode_features(test_df)
y_train_cls = train_df[TARGET_CLASSIFICATION].to_numpy().astype(int)
y_val_cls = val_df[TARGET_CLASSIFICATION].to_numpy().astype(int)
y_test_cls = test_df[TARGET_CLASSIFICATION].to_numpy().astype(int)

y_train_reg = np.log1p(np.clip(train_df[TARGET_REGRESSION].to_numpy(dtype=np.float64), 0.0, None))
y_val_reg = np.log1p(np.clip(val_df[TARGET_REGRESSION].to_numpy(dtype=np.float64), 0.0, None))

def fit_lgb(model: lgb.LGBMModel, X_tr: pd.DataFrame, y_tr: np.ndarray, X_va: pd.DataFrame, y_va: np.ndarray, rounds: int = 50) -> None:
    cb = [lgb.early_stopping(rounds, verbose=False)]
    if "eval_X" in inspect.signature(model.fit).parameters:
        model.fit(X_tr, y_tr, eval_X=X_va, eval_y=y_va, callbacks=cb)
    else:
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=cb)

def fit_calibrator(model: Any, X_va: pd.DataFrame, y_va: np.ndarray) -> CalibratedClassifierCV:
    if FrozenEstimator is not None:
        cal = CalibratedClassifierCV(estimator=FrozenEstimator(model), method="isotonic")
    else:
        cal = CalibratedClassifierCV(estimator=model, cv="prefit", method="isotonic")
    cal.fit(X_va, y_va)
    return cal

In [ ]:
N_ESTIMATORS = 1500
STOPPING_ROUNDS = 50

lgb_grid = [
    {"num_leaves": 31, "learning_rate": 0.05, "min_child_samples": 20, "reg_lambda": 1.0},
    {"num_leaves": 63, "learning_rate": 0.03, "min_child_samples": 50, "reg_lambda": 5.0},
    {"num_leaves": 127, "learning_rate": 0.03, "min_child_samples": 100, "reg_lambda": 10.0},
]
xgb_grid = [
    {"max_depth": 6, "learning_rate": 0.05, "min_child_weight": 5, "reg_lambda": 1.0},
    {"max_depth": 8, "learning_rate": 0.03, "min_child_weight": 20, "reg_lambda": 5.0},
    {"max_depth": 5, "learning_rate": 0.03, "min_child_weight": 50, "reg_lambda": 10.0},
]

best_lgb_clf, best_lgb_pr, best_lgb_cls_params = None, -1.0, {}
for params in lgb_grid:
    cand = lgb.LGBMClassifier(
        n_estimators=N_ESTIMATORS, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1, **params,
    )
    fit_lgb(cand, X_train, y_train_cls, X_val, y_val_cls, STOPPING_ROUNDS)
    val_pr = average_precision_score(y_val_cls, cand.predict_proba(X_val)[:, 1])
    logger.info("LGB Clf candidate %s -> val PR-AUC=%.4f", params, val_pr)
    if val_pr > best_lgb_pr:
        best_lgb_pr, best_lgb_clf, best_lgb_cls_params = val_pr, cand, dict(params)

best_xgb_clf, best_xgb_pr, best_xgb_cls_params = None, -1.0, {}
for params in xgb_grid:
    cand = xgb.XGBClassifier(
        n_estimators=N_ESTIMATORS, subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", early_stopping_rounds=STOPPING_ROUNDS,
        tree_method="hist", random_state=42, n_jobs=-1, **params,
    )
    cand.fit(X_train, y_train_cls, eval_set=[(X_val, y_val_cls)], verbose=False)
    val_pr = average_precision_score(y_val_cls, cand.predict_proba(X_val)[:, 1])
    logger.info("XGB Clf candidate %s -> val PR-AUC=%.4f", params, val_pr)
    if val_pr > best_xgb_pr:
        best_xgb_pr, best_xgb_clf, best_xgb_cls_params = val_pr, cand, dict(params)

lgb_cal = fit_calibrator(best_lgb_clf, X_val, y_val_cls)
xgb_cal = fit_calibrator(best_xgb_clf, X_val, y_val_cls)

best_lgb_reg, best_lgb_rmse, best_lgb_reg_params = None, float("inf"), {}
for params in lgb_grid:
    cand = lgb.LGBMRegressor(
        n_estimators=N_ESTIMATORS, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1, **params,
    )
    fit_lgb(cand, X_train, y_train_reg, X_val, y_val_reg, STOPPING_ROUNDS)
    val_rmse = float(np.sqrt(mean_squared_error(y_val_reg, cand.predict(X_val))))
    logger.info("LGB Reg candidate %s -> val log-RMSE=%.4f", params, val_rmse)
    if val_rmse < best_lgb_rmse:
        best_lgb_rmse, best_lgb_reg, best_lgb_reg_params = val_rmse, cand, dict(params)

best_xgb_reg, best_xgb_rmse, best_xgb_reg_params = None, float("inf"), {}
for params in xgb_grid:
    cand = xgb.XGBRegressor(
        n_estimators=N_ESTIMATORS, subsample=0.8, colsample_bytree=0.8,
        eval_metric="rmse", early_stopping_rounds=STOPPING_ROUNDS,
        tree_method="hist", random_state=42, n_jobs=-1, **params,
    )
    cand.fit(X_train, y_train_reg, eval_set=[(X_val, y_val_reg)], verbose=False)
    val_rmse = float(np.sqrt(mean_squared_error(y_val_reg, cand.predict(X_val))))
    logger.info("XGB Reg candidate %s -> val log-RMSE=%.4f", params, val_rmse)
    if val_rmse < best_xgb_rmse:
        best_xgb_rmse, best_xgb_reg, best_xgb_reg_params = val_rmse, cand, dict(params)

In [ ]:
def extract_best_iteration(model: Any) -> int | None:
    best_iter = getattr(model, "best_iteration_", None)
    if best_iter is None:
        best_iter = getattr(model, "best_iteration", None)
    return int(best_iter) if best_iter is not None else None

models_bundle = {
    "lgbm": {"clf": best_lgb_clf, "cal": lgb_cal, "reg": best_lgb_reg},
    "xgb": {"clf": best_xgb_clf, "cal": xgb_cal, "reg": best_xgb_reg},
}
splits = {"val": (val_df, X_val), "test": (test_df, X_test)}

metrics: dict[str, Any] = {
    "meta": {
        "n_train": int(len(train_df)),
        "n_val": int(len(val_df)),
        "n_test": int(len(test_df)),
        "n_estimators": int(N_ESTIMATORS),
        "features": list(FEATURE_COLS),
        "calibration_method": "isotonic",
        "val_calibrated_in_sample": True,
        "best_params": {
            "lgb_cls": best_lgb_cls_params,
            "xgb_cls": best_xgb_cls_params,
            "lgb_reg": best_lgb_reg_params,
            "xgb_reg": best_xgb_reg_params,
        },
        "best_iteration": {
            "lgb_cls": extract_best_iteration(best_lgb_clf),
            "xgb_cls": extract_best_iteration(best_xgb_clf),
            "lgb_reg": extract_best_iteration(best_lgb_reg),
            "xgb_reg": extract_best_iteration(best_xgb_reg),
        },
    }
}
test_raw_probs, test_cal_probs, test_pred_days = {}, {}, {}

for model_name, bundle in models_bundle.items():
    metrics[model_name] = {}
    for split_name, (split_df, X_sp) in splits.items():
        y_true = split_df[TARGET_CLASSIFICATION].to_numpy().astype(int)
        amts = split_df["invoice_amount"].to_numpy(dtype=np.float64)
        raw_p = np.asarray(bundle["clf"].predict_proba(X_sp)[:, 1], dtype=np.float64)
        cal_p = np.asarray(bundle["cal"].predict_proba(X_sp)[:, 1], dtype=np.float64)
        uncal_brier = float(brier_score_loss(y_true, raw_p))
        cls_eval = evaluate_classification(y_true, cal_p, amts)
        cls_eval["uncalibrated_brier_score"] = uncal_brier
        cls_eval["brier_improvement"] = float(uncal_brier - cls_eval["brier_score"])
        pred_d = inverse_transform_days(bundle["reg"].predict(X_sp))
        reg_eval = evaluate_regression_and_cashflow(split_df, pred_d)
        metrics[model_name][split_name] = {
            "n_samples": len(split_df),
            "classification": cls_eval,
            "regression": reg_eval,
        }
        if split_name == "test":
            test_raw_probs[model_name] = raw_p
            test_cal_probs[model_name] = cal_p
            test_pred_days[model_name] = pred_d

summary_rows = []
for model_name in ("lgbm", "xgb"):
    c = metrics[model_name]["test"]["classification"]
    r = metrics[model_name]["test"]["regression"]
    summary_rows.append({
        "model": model_name.upper(),
        "test_pr_auc": c["pr_auc"],
        "test_roc_auc": c["roc_auc"],
        "test_brier_cal": c["brier_score"],
        "test_brier_raw": c["uncalibrated_brier_score"],
        "lift_10pct": c["lift_at_k"]["0.1"],
        "late_val_cap_10pct": c["late_value_captured_at_k"]["0.1"],
        "test_mae_days": r["mae"],
        "test_median_ae_days": r["median_ae"],
        "cashflow_30d_mape_pct": r["cashflow_horizons"]["30d"]["mape_pct"],
    })
summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
# 1. SHAP Summary Plot on 1,000-row Test Sample
X_shap = X_test.sample(n=min(1000, len(X_test)), random_state=42)
explainer = shap.TreeExplainer(best_lgb_clf)
raw_shap = explainer(X_shap).values
if isinstance(raw_shap, list):
    shap_mat = np.asarray(raw_shap[1] if len(raw_shap) > 1 else raw_shap[0], dtype=np.float64)
else:
    shap_mat = np.asarray(raw_shap, dtype=np.float64)
if shap_mat.ndim == 3:
    shap_mat = shap_mat[:, :, 1] if shap_mat.shape[2] > 1 else shap_mat[:, :, 0]

metrics["shap_top_features"] = sorted(
    [{"feature": c, "mean_abs_shap": float(v)} for c, v in zip(X_shap.columns, np.mean(np.abs(shap_mat), axis=0))],
    key=lambda d: d["mean_abs_shap"],
    reverse=True,
)
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_mat, X_shap, show=False, max_display=20)
plt.title("LightGBM Late-Payment Risk SHAP Summary")
plt.tight_layout()
plt.savefig(OUT_DIR / "shap_summary.png", dpi=150, bbox_inches="tight")
plt.close("all")

# 2. Calibration Curve
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
for name, color in (("lgbm", "#1f77b4"), ("xgb", "#d62728")):
    raw_p, cal_p = test_raw_probs[name], test_cal_probs[name]
    f_r, m_r = calibration_curve(y_test_cls, raw_p, n_bins=10, strategy="quantile")
    f_c, m_c = calibration_curve(y_test_cls, cal_p, n_bins=10, strategy="quantile")
    ax.plot(m_r, f_r, marker="o", linestyle=":", alpha=0.6, color=color, label=f"{name.upper()} raw ({brier_score_loss(y_test_cls, raw_p):.4f})")
    ax.plot(m_c, f_c, marker="s", linewidth=2.0, color=color, label=f"{name.upper()} cal ({brier_score_loss(y_test_cls, cal_p):.4f})")
ax.set_title("Test Calibration Curve (Raw vs Isotonic)")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Observed Fraction Late")
ax.legend(loc="upper left", fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "calibration_curve.png", dpi=150)
plt.close(fig)

# 3. Precision-Recall Curve
fig, ax = plt.subplots(figsize=(7, 5))
base_rate = float(np.mean(y_test_cls))
ax.axhline(base_rate, linestyle="--", color="gray", label=f"Baseline ({base_rate:.3f})")
for name, color in (("lgbm", "#1f77b4"), ("xgb", "#d62728")):
    prec, rec, _ = precision_recall_curve(y_test_cls, test_cal_probs[name])
    ap = average_precision_score(y_test_cls, test_cal_probs[name])
    ax.plot(rec, prec, linewidth=2.0, color=color, label=f"{name.upper()} (PR-AUC={ap:.4f})")
ax.set_title("Test Precision-Recall Curve")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "pr_curve.png", dpi=150)
plt.close(fig)

# 4. Cashflow Forecast Plot
lgb_cf = metrics["lgbm"]["test"]["regression"]["cashflow_horizons"]
xgb_cf = metrics["xgb"]["test"]["regression"]["cashflow_horizons"]
horizons = list(lgb_cf.keys())
x = np.arange(len(horizons))
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - 0.25, [lgb_cf[h]["actual_outflow"] for h in horizons], 0.25, label="Actual", color="#2f4b7c")
ax.bar(x, [lgb_cf[h]["predicted_outflow"] for h in horizons], 0.25, label="LightGBM", color="#1f77b4")
ax.bar(x + 0.25, [xgb_cf[h]["predicted_outflow"] for h in horizons], 0.25, label="XGBoost", color="#d62728")
ax.set_xticks(x)
ax.set_xticklabels(horizons)
ax.set_title("Test Cash Outflow Forecast by Horizon")
ax.set_ylabel("Cumulative Cash Outflow ($)")
ax.legend(loc="upper left", fontsize=9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "cashflow_forecast.png", dpi=150)
plt.close(fig)

In [ ]:
preserved_cols = [
    c for c in (
        "invoice_number", "supplier_number", "operating_unit", "primary_department",
        "invoice_date", "payment_date", "invoice_amount", "days_to_pay", "is_late",
    ) if c in test_df.columns
]
preds_df = test_df[preserved_cols].copy()
preds_df["lgb_prob_late"] = test_cal_probs["lgbm"]
preds_df["xgb_prob_late"] = test_cal_probs["xgb"]
preds_df["lgb_pred_days"] = test_pred_days["lgbm"]
preds_df["xgb_pred_days"] = test_pred_days["xgb"]
preds_df.to_parquet(OUT_DIR / "test_predictions.parquet", index=False, compression="snappy")

for fname, obj in (
    ("ordinal_encoder.joblib", encoder),
    ("lgb_classifier.joblib", best_lgb_clf),
    ("lgb_calibrator.joblib", lgb_cal),
    ("xgb_classifier.joblib", best_xgb_clf),
    ("xgb_calibrator.joblib", xgb_cal),
    ("lgb_regressor.joblib", best_lgb_reg),
    ("xgb_regressor.joblib", best_xgb_reg),
):
    joblib.dump(obj, OUT_DIR / fname)

with open(OUT_DIR / "gbdt_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

logger.info("All Kaggle GBDT artifacts exported to %s", OUT_DIR)